# LULC Classification: Uncertainty Quantification Pipeline
**Project:** Multispectral Image Analysis & Uncertainty Quantification  
**Author:** Danesh Selwal  
**Date:** 2026-05-02

---
## Executive Summary
This notebook evaluates post-hoc uncertainty over pretrained LULC classifiers and exports quantitative and spatial outputs for analysis.

**Objective:**
Apply uncertainty estimation methods, compare model behavior, and export reproducible artifacts to esults/ for reporting.

---
## 1. Environment Setup & Configuration
Import dependencies, configure runtime paths, and initialize reproducibility settings before running evaluation.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

MODULE_NAME = 'sacp'
REPO_ROOT = Path("/content/drive/MyDrive/hisar_3mts_uncertainty_quantification")
MODULE_DIR = REPO_ROOT / MODULE_NAME
RESULTS_DIR = MODULE_DIR / 'results'
MODELS_DIR = MODULE_DIR / 'models'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Module: {MODULE_NAME}')
print(f'Output Directory: {RESULTS_DIR}')


In [2]:
from pathlib import Path
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow C++ INFO/WARNING logs

# Setup (Colab)
import os
import sys
import subprocess



Mounted at /content/drive


In [3]:

import io
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from tqdm.auto import tqdm
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from openpyxl import load_workbook

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)




Python: 3.12.12
TensorFlow: 2.19.0


In [4]:

# -----------------------------
# Configuration
# -----------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = REPO_ROOT
DATA_DIR = REPO_ROOT / "data"
MODEL_DIR = MODELS_DIR
OUTPUT_DIR = RESULTS_DIR
PLOT_DIR = OUTPUT_DIR / 'plots_sacp_5'

PLOT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_DIR / "hisar_3.tif"
LABEL_FILE = DATA_DIR / "clsimg_3.tif"

MODEL_FILES = {
    'AlexNet_CNN': MODEL_DIR / 'AlexNet_CNN_best.keras',
    'GFNet': MODEL_DIR / 'GFNet_best.keras',
    'ViT_UNet': MODEL_DIR / 'ViT_UNet_best.keras',
}
MODEL_NAME_MAP = {
    'AlexNet_CNN': 'AlexNet',
    'GFNet': 'GFNet',
    'ViT_UNet': 'ViT',
}

TRUSTED_MODEL_ROOTS = [
    MODELS_DIR,
    MODELS_DIR,
]

# Data geometry for 6-band setup
H, W, B = 330, 307, 6
PATCH_SIZE = 9
TRAIN_PERCENT = 0.75
CALIB_FRACTION_OF_TEST = 0.5

# SACP hyperparameters
SACP_ALPHA = 0.05
SACP_LAMBDA = 0.5
SACP_K = 1
SACP_WINDOW_SIZE = 7

BATCH_SIZE = 128
EPS = 1e-12

# Outputs
EXCEL_PATH = OUTPUT_DIR / 'conformal_reports_SACP_7_all_models.xlsx'
SUMMARY_CSV_PATH = OUTPUT_DIR / 'summary_sacp_7_metrics.csv'
PER_CLASS_CSV_PATH = OUTPUT_DIR / 'per_class_sacp_7_coverage.csv'
RUN_CONFIG_JSON_PATH = OUTPUT_DIR / 'run_config_sacp_7.json'

for k, v in MODEL_FILES.items():
    print(f'{k}: {v}')
print('Excel path:', EXCEL_PATH)




AlexNet_CNN: /content/drive/My Drive/m_p/saved_models/AlexNet_CNN_best.keras
GFNet: /content/drive/My Drive/m_p/saved_models/GFNet_best.keras
ViT_UNet: /content/drive/My Drive/m_p/saved_models/ViT_UNet_best.keras
Excel path: /content/drive/My Drive/m_p/uncertainty_results/conformal_reports_SACP_7_all_models.xlsx


In [5]:

# -----------------------------
# Guards
# -----------------------------
def assert_environment_and_files():
    if not DATA_FILE.exists() or not LABEL_FILE.exists():
        raise FileNotFoundError(
            f'Data files missing:\n- {DATA_FILE}\n- {LABEL_FILE}'
        )
    missing = [str(p) for p in MODEL_FILES.values() if not p.exists()]
    if missing:
        raise FileNotFoundError('Missing model files:\n' + '\n'.join(missing))

assert_environment_and_files()
print('All required files found.')




All required files found.


## 2. Data Ingestion & Preprocessing
Load multispectral inputs and reference labels, apply normalization, and prepare patch-based tensors for model training/evaluation.


In [6]:

# -----------------------------
# Data pipeline with coordinate tracking
# -----------------------------
def extract_labeled_patches_with_coords(x_img, y_img, patch_size=9):
    pad = patch_size // 2
    x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode='edge')

    coords = np.argwhere(y_img > 0)
    x_patches = np.empty((coords.shape[0], patch_size, patch_size, x_img.shape[-1]), dtype=np.float32)
    y_labels = np.empty((coords.shape[0],), dtype=np.int32)

    for i, (r, c) in enumerate(coords):
        x_patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        y_labels[i] = int(y_img[r, c]) - 1

    return x_patches, y_labels, coords


def split_calib_eval_with_coords(x_test, y_test, coords_test, seed=42, calib_fraction=0.5):
    test_size = 1.0 - calib_fraction
    try:
        x_cal, x_eval, y_cal, y_eval, coords_cal, coords_eval = train_test_split(
            x_test, y_test, coords_test,
            test_size=test_size,
            random_state=seed,
            stratify=y_test,
        )
    except ValueError:
        x_cal, x_eval, y_cal, y_eval, coords_cal, coords_eval = train_test_split(
            x_test, y_test, coords_test,
            test_size=test_size,
            random_state=seed,
            stratify=None,
        )
    return x_cal, x_eval, y_cal, y_eval, coords_cal, coords_eval



# -----------------------------
# Generalized Data Loading
# -----------------------------
import scipy.io as sio
import pandas as pd
import numpy as np

def universal_load_data(data_path, label_path):
    data_path = str(data_path)
    label_path = str(label_path)
    
    # Load features
    if data_path.endswith('.mat'):
        mat = sio.loadmat(data_path)
        x = next(v for k, v in mat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif data_path.endswith('.csv'):
        x = pd.read_csv(data_path).to_numpy(dtype=np.float32)
    elif data_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(data_path) as src:
                x = src.read()
                x = np.moveaxis(x, 0, -1)
        except ImportError:
            print("rasterio not installed. Using dummy data.")
            x = np.zeros((10,10,3))

    # Load labels
    if label_path.endswith('.mat'):
        lmat = sio.loadmat(label_path)
        y = next(v for k, v in lmat.items() if not k.startswith('__') and isinstance(v, np.ndarray))
    elif label_path.endswith('.csv'):
        y = pd.read_csv(label_path).to_numpy(dtype=np.int32)
    elif label_path.endswith(('.tif', '.tiff')):
        try:
            import rasterio
            with rasterio.open(label_path) as src:
                y = src.read(1)
        except ImportError:
            y = np.zeros((10,10))

    # Normalization for 3D tensors
    if len(x.shape) == 3:
        x_norm = np.empty_like(x, dtype=np.float32)
        for b_idx in range(x.shape[-1]):
            band = x[:, :, b_idx]
            b_min, b_max = np.min(band), np.max(band)
            x_norm[:, :, b_idx] = (band - b_min) / max(b_max - b_min, 1e-8)
        x = x_norm
        
    return x, y

# Apply Generalized Loader
x_img, y_img = universal_load_data(DATA_FILE, LABEL_FILE)

# Handle flat CSVs by requesting user input or fallback
if len(x_img.shape) == 3:
    H, W, B = x_img.shape
else:
    print("WARNING: Data is flat. Please manually reshape x_img and y_img, then define H, W, B.")
    H, W, B = 330, 307, 6 # Default fallback for flat data

num_classes = int(np.unique(y_img).size)
print(f"Loaded Data Shape: {x_img.shape}, Labels Shape: {y_img.shape}, Classes: {num_classes}")

# Dynamic Color Palette Setup
import seaborn as sns
from matplotlib.colors import ListedColormap
BACKGROUND_COLOR = "#000000"
CLASS_COLOR_BASE = sns.color_palette("hls", max(10, num_classes)).as_hex()
X_all, y_all, coords_all = extract_labeled_patches_with_coords(x_img, y_img, PATCH_SIZE)
num_classes = int(np.unique(y_all).size)

_, x_test_pool, _, y_test_pool, _, coords_test_pool = train_test_split(
    X_all, y_all, coords_all,
    train_size=TRAIN_PERCENT,
    random_state=SEED,
    stratify=y_all,
)

x_cal, x_eval, y_cal, y_eval, coords_cal, coords_eval = split_calib_eval_with_coords(
    x_test_pool,
    y_test_pool,
    coords_test_pool,
    seed=SEED,
    calib_fraction=CALIB_FRACTION_OF_TEST,
)

print('x_img:', x_img.shape, 'y_img:', y_img.shape)
print('X_all:', X_all.shape, 'num_classes:', num_classes)
print('x_cal:', x_cal.shape, 'x_eval:', x_eval.shape)
print('coords_cal:', coords_cal.shape, 'coords_eval:', coords_eval.shape)




x_img: (330, 307, 6) y_img: (330, 307)
X_all: (17239, 9, 9, 6) num_classes: 7
x_cal: (2155, 9, 9, 6) x_eval: (2155, 9, 9, 6)
coords_cal: (2155, 2) coords_eval: (2155, 2)


In [7]:

# -----------------------------
# Custom objects for model loading
# -----------------------------
@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        b = tf.shape(images)[0]
        n = tf.shape(patches)[1] * tf.shape(patches)[2]
        d = tf.shape(patches)[-1]
        return tf.reshape(patches, [b, n, d])

    def get_config(self):
        c = super().get_config()
        c.update({'patch_size': self.patch_size})
        return c


@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patches):
        pos = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(pos)

    def get_config(self):
        c = super().get_config()
        c.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return c


@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(name='w_real', shape=(self.token_side, self.token_side, channels), initializer='glorot_uniform', trainable=True)
        self.w_imag = self.add_weight(name='w_imag', shape=(self.token_side, self.token_side, channels), initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, x):
        b = tf.shape(x)[0]
        c = tf.shape(x)[-1]
        x2 = tf.reshape(x, [b, self.token_side, self.token_side, c])
        x_fft = tf.signal.fft2d(tf.cast(x2, tf.complex64))
        w = tf.complex(self.w_real, self.w_imag)
        x_i = tf.math.real(tf.signal.ifft2d(x_fft * w))
        return tf.reshape(x_i, [b, self.token_side * self.token_side, c])

    def get_config(self):
        c = super().get_config()
        c.update({'token_side': self.token_side})
        return c


@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches + 1, output_dim=projection_dim)

    def build(self, input_shape):
        self.cls_token = self.add_weight(name='cls_token', shape=(1, 1, self.projection_dim), initializer='zeros', trainable=True)
        super().build(input_shape)

    def call(self, patches):
        b = tf.shape(patches)[0]
        p = self.projection(patches)
        cls = tf.repeat(self.cls_token, repeats=b, axis=0)
        x = tf.concat([cls, p], axis=1)
        pos = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(pos)

    def get_config(self):
        c = super().get_config()
        c.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return c


CUSTOM_OBJECTS = {
    'PatchExtractor': PatchExtractor,
    'PatchPositionEncoder': PatchPositionEncoder,
    'GlobalFilterLayer': GlobalFilterLayer,
    'PatchEncoderWithCLS': PatchEncoderWithCLS,
}




In [8]:

# -----------------------------
# Secure model loading + smoke check
# -----------------------------
def is_trusted_model_path(path: Path) -> bool:
    p = path.expanduser().resolve()
    for root in TRUSTED_MODEL_ROOTS:
        r = root.expanduser().resolve()
        if p == r or r in p.parents:
            return True
    return False


def load_models(model_files, custom_objects):
    loaded = {}
    for key, path in model_files.items():
        path = Path(path)
        print(f'Loading {key} from {path}')

        if not is_trusted_model_path(path):
            raise RuntimeError(f'Untrusted model path: {path}')

        first_err = None
        try:
            m = keras.models.load_model(path, custom_objects=custom_objects, compile=False, safe_mode=False)
            loaded[key] = m
            print(f'  Loaded {key} successfully.')
            continue
        except Exception as e:
            first_err = e
            print(f'  Primary load failed for {key}: {e}')
            print('  Suggested checks: file integrity, custom_objects compatibility, TF/Keras versions.')

        lambda_related = 'lambda' in str(first_err).lower() or 'safe_mode' in str(first_err).lower()
        if lambda_related:
            try:
                keras.config.enable_unsafe_deserialization()
                m = keras.models.load_model(path, custom_objects=custom_objects, compile=False, safe_mode=False)
                loaded[key] = m
                print(f'  Fallback load succeeded for {key}.')
                continue
            except Exception as second_err:
                raise RuntimeError(
                    f'Failed to load {key} from {path}.\n'
                    f'Primary error: {first_err}\nFallback error: {second_err}'
                )

        raise RuntimeError(f'Failed to load {key} from {path}. Error: {first_err}')

    return loaded


models = load_models(MODEL_FILES, CUSTOM_OBJECTS)

x_smoke = x_eval[:8]
for key, model in models.items():
    p = model.predict(x_smoke, verbose=0)
    assert p.ndim == 2, f'{key}: expected rank-2 output, got {p.shape}'
    assert p.shape[1] == num_classes, f'{key}: expected class dim {num_classes}, got {p.shape[1]}'
    assert np.isfinite(p).all(), f'{key}: NaN/Inf in prediction output'
    print(key, 'smoke output shape:', p.shape)




Loading AlexNet_CNN from /content/drive/My Drive/m_p/saved_models/AlexNet_CNN_best.keras
  Loaded AlexNet_CNN successfully.
Loading GFNet from /content/drive/My Drive/m_p/saved_models/GFNet_best.keras
  Loaded GFNet successfully.
Loading ViT_UNet from /content/drive/My Drive/m_p/saved_models/ViT_UNet_best.keras
  Loaded ViT_UNet successfully.
AlexNet_CNN smoke output shape: (8, 7)
GFNet smoke output shape: (8, 7)
ViT_UNet smoke output shape: (8, 7)


In [9]:

# -----------------------------
# SACP utilities and plotting
# -----------------------------
CLASS_COLORS_BASE = ['#0000FF', '#00FF00', '#FF0000', '#00FFFF', '#FF00FF', '#FFFF00', '#A52A2A', '#FFA500', '#7FFF00', '#8A2BE2']
UNCERTAIN_COLOR = '#808080'


def get_class_colors(n):
    if n <= len(CLASS_COLORS_BASE):
        return CLASS_COLORS_BASE[:n]
    colors = CLASS_COLORS_BASE.copy()
    cmap = plt.cm.get_cmap('tab20', n)
    for i in range(len(colors), n):
        c = cmap(i)
        colors.append('#%02x%02x%02x' % (int(c[0]*255), int(c[1]*255), int(c[2]*255)))
    return colors[:n]


def normalize_probs(prob, eps=1e-12):
    prob = np.asarray(prob, dtype=np.float64)
    prob = np.nan_to_num(prob, nan=0.0, posinf=0.0, neginf=0.0)
    prob = np.clip(prob, 0.0, 1.0)
    rs = prob.sum(axis=-1, keepdims=True)
    rs = np.where(rs <= eps, 1.0, rs)
    return prob / rs


def predict_probs(model, x, batch_size=128):
    return normalize_probs(model.predict(x, batch_size=batch_size, verbose=0), eps=EPS)


def compute_set_metrics(pred_sets, y_true):
    pred_sets = pred_sets.astype(bool)
    set_sizes = pred_sets.sum(axis=1)
    covered = pred_sets[np.arange(len(y_true)), y_true].astype(int)
    return {
        'empirical_coverage': float(np.mean(covered)),
        'avg_set_size': float(np.mean(set_sizes)),
        'median_set_size': float(np.median(set_sizes)),
        'singleton_rate': float(np.mean(set_sizes == 1)),
        'empty_set_rate': float(np.mean(set_sizes == 0)),
        'set_sizes': set_sizes,
        'covered': covered,
    }


def per_class_coverage_df(pred_sets, y_true, n_classes):
    rows = []
    for c in range(n_classes):
        mask = (y_true == c)
        support = int(mask.sum())
        cov = float(np.mean(pred_sets[mask, c])) if support > 0 else np.nan
        rows.append({'class_id': c, 'class_coverage': cov, 'support_count': support})
    return pd.DataFrame(rows)


def fig_to_buffer(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    return buf


def add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01):
    ymax = max(ax.get_ylim()[1], 1e-9)
    for p in ax.patches:
        h = p.get_height()
        if np.isnan(h):
            continue
        ax.text(p.get_x() + p.get_width()/2, h + y_pad * ymax, fmt.format(h), ha='center', va='bottom', fontsize=9)


def make_per_class_coverage_plot(per_cls_df, alpha, title):
    fig, ax = plt.subplots(figsize=(15, 7))
    labels = [f'Class {int(c)}' for c in per_cls_df['class_id']]
    vals = per_cls_df['class_coverage'].to_numpy(dtype=float)
    ax.bar(labels, vals, edgecolor='black', color='skyblue')
    ax.axhline(1 - alpha, color='red', linestyle='--', linewidth=2, label=f'Desired Coverage ({1-alpha:.2f})')
    ax.set_title(title, fontsize=16)
    ax.set_xlabel('Class')
    ax.set_ylabel('Coverage')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.tick_params(axis='x', rotation=45)
    add_bar_labels(ax, fmt='{:.2f}', y_pad=0.01)
    ax.legend(loc='upper right')
    fig.tight_layout()
    return fig_to_buffer(fig)


def make_certain_uncertain_map_plot(set_sizes_map, title):
    disp = np.where(set_sizes_map == 1, 0, 1)
    cmap = ListedColormap(['#FFFF00', '#001F3F'])
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(disp, cmap=cmap)
    ax.set_title(title, fontsize=16)
    ax.axis('off')

    legend_handles = [
        Patch(facecolor='#FFFF00', edgecolor='black', label='Certain (Size=1)'),
        Patch(facecolor='#001F3F', edgecolor='black', label='Uncertain (Size!=1)'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0.0, frameon=True)
    fig.tight_layout()
    return fig_to_buffer(fig)


def make_masked_class_map_plot(combined_map, n_classes, title):
    cmap = ListedColormap(get_class_colors(n_classes) + [UNCERTAIN_COLOR])
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(combined_map, cmap=cmap, vmin=0, vmax=n_classes)
    ax.set_title(title, fontsize=16)
    ax.axis('off')

    cbar = fig.colorbar(im, ax=ax, ticks=np.arange(n_classes + 1), fraction=0.046, pad=0.04)
    cbar.set_ticklabels([f'Class {i}' for i in range(n_classes)] + ['Uncertain'])
    fig.tight_layout()
    return fig_to_buffer(fig)


def make_pixel_count_plot(pixel_counts_df, title, n_classes):
    colors = get_class_colors(n_classes) + [UNCERTAIN_COLOR]
    fig, ax = plt.subplots(figsize=(10, 6))
    labels = pixel_counts_df['label'].tolist()
    counts = pixel_counts_df['pixel_count'].tolist()
    ax.bar(labels, counts, color=colors[:len(labels)], edgecolor='black')
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel('Number of Pixels')
    ax.set_title(title, fontsize=16)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    ymax = max(counts) if len(counts) else 1
    for i, v in enumerate(counts):
        ax.text(i, v + 0.01 * ymax, f'{int(v):,}', ha='center', va='bottom', fontsize=9)

    fig.tight_layout()
    return fig_to_buffer(fig)


def build_pixel_counts_df(combined_map, n_classes):
    uniq, cnt = np.unique(combined_map, return_counts=True)
    m = {int(k): int(v) for k, v in zip(uniq, cnt)}
    rows = []
    for c in range(n_classes):
        rows.append({'class_id': c, 'label': f'Class {c}', 'pixel_count': m.get(c, 0)})
    rows.append({'class_id': n_classes, 'label': 'Uncertain', 'pixel_count': m.get(n_classes, 0)})
    return pd.DataFrame(rows)


def predict_full_scene_probs(model, x_img, H, W, B, patch_size, batch_size=128):
    pad = patch_size // 2
    x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode='edge')

    test_patch = x_pad[0:patch_size, 0:patch_size, :][None, ...]
    n_classes = predict_probs(model, test_patch, batch_size=1).shape[1]

    prob_full = np.zeros((H, W, n_classes), dtype=np.float32)
    for col in range(W):
        patchs = np.zeros((H, patch_size, patch_size, B), dtype=np.float32)
        for row in range(H):
            patchs[row] = x_pad[row:row + patch_size, col:col + patch_size, :]
        prob_col = predict_probs(model, patchs, batch_size=batch_size)
        prob_full[:, col, :] = prob_col

        if (col + 1) % 50 == 0 or (col + 1) == W:
            print(f'  full-scene progress: {col + 1}/{W}')

    assert prob_full.shape == (H, W, n_classes)
    return prob_full


class SpatialConformalPredictor:
    def __init__(self, height, width, num_classes, lambda_=0.5, alpha=0.05, k=1, window_size=3, seed=42):
        self.H = height
        self.W = width
        self.num_classes = num_classes
        self.lmd = lambda_
        self.alpha = alpha
        self.k = k
        self.window_size = window_size
        self.seed = seed

        # Build neighbor offsets from window_size
        # e.g. window_size=3 → radius=1 → offsets in [-1, 0, 1] excluding (0,0)
        # e.g. window_size=5 → radius=2 → offsets in [-2,-1,0,1,2] excluding (0,0)
        assert window_size >= 3 and window_size % 2 == 1, \
            f"window_size must be an odd integer >= 3, got {window_size}"
        radius = window_size // 2
        self.neighbors = [
            (dr, dc)
            for dr in range(-radius, radius + 1)
            for dc in range(-radius, radius + 1)
            if not (dr == 0 and dc == 0)
        ]

    def compute_aps_scores(self, probabilities, labels=None):
        n = probabilities.shape[0]
        sorted_indices = np.argsort(probabilities, axis=1)[:, ::-1]
        sorted_probs = np.take_along_axis(probabilities, sorted_indices, axis=1)
        cumsum = np.cumsum(sorted_probs, axis=1)

        rng = np.random.default_rng(self.seed)
        U = rng.random(n)

        if labels is not None:
            scores = np.zeros(n)
            for i in range(n):
                y = int(labels[i])
                rank = int(np.where(sorted_indices[i] == y)[0][0])
                if rank == 0:
                    scores[i] = U[i] * sorted_probs[i, 0]
                else:
                    scores[i] = cumsum[i, rank - 1] + U[i] * sorted_probs[i, rank]
            return scores

        scores_matrix = np.zeros_like(probabilities)
        for i in range(n):
            scores_sorted = np.zeros(self.num_classes)
            scores_sorted[0] = U[i] * sorted_probs[i, 0]
            scores_sorted[1:] = cumsum[i, :-1] + U[i] * sorted_probs[i, 1:]
            scores_matrix[i, sorted_indices[i]] = scores_sorted

        return scores_matrix

    def spatial_smoothing(self, score_map, mask_map):
        smoothed = np.copy(score_map)
        H, W, C = score_map.shape

        rows, cols = np.where(mask_map)
        for r, c in zip(rows, cols):
            ori = score_map[r, c]
            n_sum = np.zeros(C)
            n_count = 0

            for dr, dc in self.neighbors:
                nr, nc = r + dr, c + dc
                if 0 <= nr < H and 0 <= nc < W and mask_map[nr, nc]:
                    n_sum += score_map[nr, nc]
                    n_count += 1

            if n_count > 0:
                smoothed[r, c] = self.lmd * ori + self.lmd * (n_sum / n_count)

        return smoothed

    def fit_calibrate(self, calib_probs, calib_labels, calib_indices, test_probs, test_indices):
        calib_scores_mat = self.compute_aps_scores(calib_probs)
        test_scores_mat = self.compute_aps_scores(test_probs)

        score_map = np.zeros((self.H, self.W, self.num_classes), dtype=np.float64)
        mask_map = np.zeros((self.H, self.W), dtype=bool)

        for i, (r, c) in enumerate(calib_indices):
            score_map[r, c] = calib_scores_mat[i]
            mask_map[r, c] = True

        for i, (r, c) in enumerate(test_indices):
            score_map[r, c] = test_scores_mat[i]
            mask_map[r, c] = True

        current_map = score_map
        for _ in range(self.k):
            current_map = self.spatial_smoothing(current_map, mask_map)

        fused_calib_scores = []
        for i, (r, c) in enumerate(calib_indices):
            y = int(calib_labels[i])
            fused_calib_scores.append(current_map[r, c, y])
        fused_calib_scores = np.array(fused_calib_scores)

        n = len(fused_calib_scores)
        q_level = np.ceil((n + 1) * (1 - self.alpha)) / n
        q_level = min(1.0, max(0.0, q_level))
        q_hat = float(np.quantile(fused_calib_scores, q_level, method='higher'))

        pred_sets = np.zeros((len(test_indices), self.num_classes), dtype=bool)
        for i, (r, c) in enumerate(test_indices):
            pred_sets[i] = (current_map[r, c] <= q_hat)
            if not pred_sets[i].any():
                pred_sets[i, int(np.argmin(current_map[r, c]))] = True

        avg_size = float(pred_sets.sum(axis=1).mean())
        return pred_sets, q_hat, avg_size




In [10]:
WINDOW_SIZES = [3, 5, 7, 9]
master_summary = []

for size in WINDOW_SIZES:
    print(f'\n' + '#' * 60)
    print(f'### RUNNING SACP WITH WINDOW SIZE: {size} ###')
    print('#' * 60 + '\n')
    
    # Update paths and params for this size
    CURRENT_EXCEL_PATH = OUTPUT_DIR / f'conformal_reports_SACP_{size}_all_models.xlsx'
    CURRENT_SUMMARY_CSV = OUTPUT_DIR / f'summary_sacp_{size}_metrics.csv'
    CURRENT_PER_CLASS_CSV = OUTPUT_DIR / f'per_class_sacp_{size}_coverage.csv'
    CURRENT_RUN_CONFIG = OUTPUT_DIR / f'run_config_sacp_{size}.json'
    
    current_outputs = []
    for model_key, model in models.items():
        model_name = MODEL_NAME_MAP.get(model_key, model_key)
        print(f'\nRunning SACP for {model_name} (Size={size})...')

        out = build_sacp_outputs_for_model(
            model_name=model_name,
            model=model,
            x_cal=x_cal,
            y_cal=y_cal,
            coords_cal=coords_cal,
            x_eval=x_eval,
            y_eval=y_eval,
            coords_eval=coords_eval,
            x_img=x_img,
            alpha=SACP_ALPHA,
            lambda_=SACP_LAMBDA,
            k=SACP_K,
            window_size=size,
            batch_size=BATCH_SIZE,
        )
        current_outputs.append(out)
    
    # 3. Post-Process results for this specific size
    sz_summary_df = pd.DataFrame([o['summary'] for o in current_outputs]).sort_values('model_name').reset_index(drop=True)
    master_summary.append(sz_summary_df)
    
    # (Optional: Add Excel saving logic here if you want separate files per size)
    # For brevity, let's assume we save the summary for each run
    sz_summary_df.to_csv(CURRENT_SUMMARY_CSV, index=False)
    print(f'Saved summary to {CURRENT_SUMMARY_CSV}')

final_summary = pd.concat(master_summary, ignore_index=True)
final_summary



==================== Running SACP for AlexNet ====================
Generating full-scene probabilities for AlexNet ...
  full-scene progress: 50/307
  full-scene progress: 100/307
  full-scene progress: 150/307
  full-scene progress: 200/307
  full-scene progress: 250/307
  full-scene progress: 300/307
  full-scene progress: 307/307

==================== Running SACP for GFNet ====================
Generating full-scene probabilities for GFNet ...
  full-scene progress: 50/307
  full-scene progress: 100/307
  full-scene progress: 150/307
  full-scene progress: 200/307
  full-scene progress: 250/307
  full-scene progress: 300/307
  full-scene progress: 307/307

==================== Running SACP for ViT ====================
Generating full-scene probabilities for ViT ...
  full-scene progress: 50/307
  full-scene progress: 100/307
  full-scene progress: 150/307
  full-scene progress: 200/307
  full-scene progress: 250/307
  full-scene progress: 300/307
  full-scene progress: 307/307


,model_name,method,target_coverage,empirical_coverage,avg_set_size,median_set_size,singleton_rate,empty_set_rate,runtime_sec,alpha,lambda,k,q_hat,mean_per_class_coverage
0,AlexNet,SACP,0.95,0.998608,1.000928,1.0,0.999072,0.0,62.135974,0.05,0.5,1,0.762639,0.998012
1,GFNet,SACP,0.95,0.997680,1.000928,1.0,0.999072,0.0,57.501753,0.05,0.5,1,0.757632,0.997214
2,ViT,SACP,0.95,0.997216,1.002320,1.0,0.997680,0.0,74.594362,0.05,0.5,1,0.736656,0.997585


In [11]:

# -----------------------------
# Comparison plot across 3 models (SACP)
# -----------------------------
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

sns.barplot(data=summary_df, x='model_name', y='empirical_coverage', ax=axes[0], palette='Set2')
axes[0].axhline(1 - SACP_ALPHA, linestyle='--', color='red', linewidth=1.8, label=f'Target ({1-SACP_ALPHA:.2f})')
axes[0].set_title('Empirical Coverage')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Coverage')
axes[0].set_ylim(0, 1.1)
axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1.0), frameon=True)
add_bar_labels(axes[0], fmt='{:.2f}', y_pad=0.01)

sns.barplot(data=summary_df, x='model_name', y='avg_set_size', ax=axes[1], palette='Set3')
axes[1].set_title('Average Set Size')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Set Size')
add_bar_labels(axes[1], fmt='{:.2f}', y_pad=0.01)

sns.barplot(data=summary_df, x='model_name', y='runtime_sec', ax=axes[2], palette='Set1')
axes[2].set_title('Runtime (sec)')
axes[2].set_xlabel('Model')
axes[2].set_ylabel('Seconds')
add_bar_labels(axes[2], fmt='{:.1f}', y_pad=0.01)

mean_pc = per_class_df.groupby('model_name', as_index=False)['class_coverage'].mean().rename(columns={'class_coverage':'mean_per_class_coverage'})
sns.barplot(data=mean_pc, x='model_name', y='mean_per_class_coverage', ax=axes[3], palette='Dark2')
axes[3].set_title('Mean Per-Class Coverage')
axes[3].set_xlabel('Model')
axes[3].set_ylabel('Coverage')
axes[3].set_ylim(0, 1.1)
add_bar_labels(axes[3], fmt='{:.2f}', y_pad=0.01)

for ax in axes:
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.tight_layout()
compare_buf = fig_to_buffer(fig)




In [12]:

# -----------------------------
# Excel export
# -----------------------------
def sanitize_sheet_name(name):
    bad = ['\\', '/', '*', '?', ':', '[', ']']
    for ch in bad:
        name = name.replace(ch, '_')
    return name[:31]


def make_sheet_name(base, used):
    base = sanitize_sheet_name(base)
    candidate = base
    i = 1
    while candidate in used:
        suffix = f'_{i}'
        candidate = base[:31 - len(suffix)] + suffix
        i += 1
    used.add(candidate)
    return candidate


def insert_buffer_image(ws, row, col, img_buf, x_scale=0.8, y_scale=0.8):
    img_buf.seek(0)
    ws.insert_image(row, col, 'plot.png', {'image_data': img_buf, 'x_scale': x_scale, 'y_scale': y_scale})


def write_model_sheet(writer, workbook, output, sheet_name):
    ws = workbook.add_worksheet(sheet_name)
    writer.sheets[sheet_name] = ws

    ws.write(0, 0, f"SACP - {output['model_name']}")

    # Tables (left)
    row = 2
    for tname, tdf in output['tables'].items():
        ws.write(row, 0, tname)
        tdf.to_excel(writer, sheet_name=sheet_name, startrow=row + 1, startcol=0, index=False)
        row += len(tdf) + 4

    # Plots (right)
    img_row = 0
    img_col = 9
    for pname, pbuf in output['plot_buffers'].items():
        ws.write(img_row, img_col, pname)
        insert_buffer_image(ws, img_row + 1, img_col, pbuf, x_scale=0.75, y_scale=0.75)
        img_row += 24


run_config = {
    'seed': SEED,
    'project_root': str(PROJECT_ROOT),
    'data_file': str(DATA_FILE),
    'label_file': str(LABEL_FILE),
    'model_dir': str(MODEL_DIR),
    'output_dir': str(OUTPUT_DIR),
    'h': H,
    'w': W,
    'b': B,
    'patch_size': PATCH_SIZE,
    'train_percent': TRAIN_PERCENT,
    'calib_fraction_of_test': CALIB_FRACTION_OF_TEST,
    'alpha': SACP_ALPHA,
    'lambda': SACP_LAMBDA,
    'k': SACP_K,
    'batch_size': BATCH_SIZE,
    'timestamp_utc': pd.Timestamp.utcnow().isoformat(),
}

summary_df.to_csv(SUMMARY_CSV_PATH, index=False)
per_class_df.to_csv(PER_CLASS_CSV_PATH, index=False)
with open(RUN_CONFIG_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(run_config, f, indent=2)

used = set()
with pd.ExcelWriter(EXCEL_PATH, engine='xlsxwriter') as writer:
    wb = writer.book

    s_summary = make_sheet_name('Summary_Compact', used)
    s_cfg = make_sheet_name('Run_Config', used)
    s_cmp = make_sheet_name('Compare_SACP', used)

    summary_df.to_excel(writer, sheet_name=s_summary, index=False)
    pd.DataFrame(list(run_config.items()), columns=['key','value']).to_excel(writer, sheet_name=s_cfg, index=False)

    for out in all_outputs:
        model_name = out['model_name']
        sheet = make_sheet_name(f'SACP_{model_name}', used)
        write_model_sheet(writer, wb, out, sheet)

    ws_cmp = wb.add_worksheet(s_cmp)
    writer.sheets[s_cmp] = ws_cmp
    ws_cmp.write(0, 0, 'SACP Model Comparison')
    summary_df.to_excel(writer, sheet_name=s_cmp, startrow=2, startcol=0, index=False)
    mean_pc.to_excel(writer, sheet_name=s_cmp, startrow=2, startcol=10, index=False)

    ws_cmp.write(12, 0, 'Grouped Comparison Plot')
    insert_buffer_image(ws_cmp, 13, 0, compare_buf, x_scale=0.75, y_scale=0.75)

print('Saved workbook:', EXCEL_PATH)
print('Saved summary csv:', SUMMARY_CSV_PATH)
print('Saved per-class csv:', PER_CLASS_CSV_PATH)
print('Saved run-config json:', RUN_CONFIG_JSON_PATH)




Saved workbook: /content/drive/My Drive/m_p/uncertainty_results/conformal_reports_SACP_7_all_models.xlsx
Saved summary csv: /content/drive/My Drive/m_p/uncertainty_results/summary_sacp_7_metrics.csv
Saved per-class csv: /content/drive/My Drive/m_p/uncertainty_results/per_class_sacp_7_coverage.csv
Saved run-config json: /content/drive/My Drive/m_p/uncertainty_results/run_config_sacp_7.json


In [13]:

# -----------------------------
# Final validation
# -----------------------------
assert EXCEL_PATH.exists(), f'Workbook not found: {EXCEL_PATH}'

wb = load_workbook(EXCEL_PATH, read_only=True)
sheets = set(wb.sheetnames)
required_prefixes = {
    'Summary_Compact',
    'Run_Config',
    'SACP_AlexNet',
    'SACP_GFNet',
    'SACP_ViT',
    'Compare_SACP',
}
for req in required_prefixes:
    assert any(s.startswith(req) for s in sheets), f'Missing sheet: {req}'

assert len(summary_df) == 3, f'Expected 3 rows (3 models), got {len(summary_df)}'
assert ((summary_df['empirical_coverage'] >= 0) & (summary_df['empirical_coverage'] <= 1)).all()

for out in all_outputs:
    px = out['tables']['Pixel Counts']['pixel_count'].sum()
    assert int(px) == H * W, f"Pixel-count mismatch for {out['model_name']}: {px} vs {H*W}"

print('Validation passed.')
summary_df




Validation passed.


,model_name,method,target_coverage,empirical_coverage,avg_set_size,median_set_size,singleton_rate,empty_set_rate,runtime_sec,alpha,lambda,k,q_hat,mean_per_class_coverage
0,AlexNet,SACP,0.95,0.998608,1.000928,1.0,0.999072,0.0,62.135974,0.05,0.5,1,0.762639,0.998012
1,GFNet,SACP,0.95,0.997680,1.000928,1.0,0.999072,0.0,57.501753,0.05,0.5,1,0.757632,0.997214
2,ViT,SACP,0.95,0.997216,1.002320,1.0,0.997680,0.0,74.594362,0.05,0.5,1,0.736656,0.997585
